In [2]:
import supy
from schema.dev.def_config_suews import *
import pandas as pd
import yaml

supy.show_version()

SuPy version: 2024.8.2.dev82
-------------


In [5]:
suews_config=SUEWSConfig()
import yaml

df_state_init, df_forcing = supy.load_SampleData()

# Convert model dump to YAML format
yaml_output = yaml.dump(suews_config.model_dump(), sort_keys=False, allow_unicode=True)
with open('config-suews-output.yml', 'w') as file:
    yaml.dump(suews_config.model_dump(), file, sort_keys=False, allow_unicode=True)

suews_config_load_back=SUEWSConfig(**yaml.safe_load(yaml_output))

df_state_test=suews_config_load_back.to_df_state()

supy.run_supy(df_forcing, df_state_test)



2024-12-04 09:50:26,564 - SuPy - INFO - All cache cleared.
2024-12-04 09:50:28,312 - SuPy - INFO - ====================
2024-12-04 09:50:28,312 - SuPy - INFO - Simulation period:
2024-12-04 09:50:28,313 - SuPy - INFO -   Start: 2012-01-01 00:05:00
2024-12-04 09:50:28,313 - SuPy - INFO -   End: 2013-01-01 00:00:00
2024-12-04 09:50:28,313 - SuPy - INFO - 
2024-12-04 09:50:28,314 - SuPy - INFO - No. of grids: 1
2024-12-04 09:50:28,314 - SuPy - INFO - SuPy is running in serial mode


ValueError: cannot reshape array of size 48 into shape (10,2)

In [ ]:
supy.run_supy(forci)

In [11]:
df_state_init, df_forcing = supy.load_SampleData()
df_forcing=df_forcing.loc['2012'].iloc[1:]

suews_config_yml_path = './schema/dev/config-suews_update.yml'
# config = SUEWSConfig(
#             name="test config",
#             description="test configuration for roundtrip testing",
#             site=[Site()],
# )
# df_state = config.to_df_state()

suews_config_yml = yaml.safe_load(open(suews_config_yml_path, 'r'))
config = SUEWSConfig(**suews_config_yml)
config_dict = config.dict()

class MyDumper(yaml.SafeDumper):
    def __init__(self, *args, **kwargs):
        super(MyDumper, self).__init__(*args, **kwargs)

    def increase_indent(self, flow=False, indentless=False):
        return super(MyDumper, self).increase_indent(flow, False)

def represent_list_indented(dumper, data):
    return dumper.represent_sequence('tag:yaml.org,2002:seq', data, flow_style=False)

yaml.add_representer(list, represent_list_indented, Dumper=MyDumper)

with open('./output.yml', 'w') as file:
    yaml.dump(config_dict, file, Dumper=MyDumper, default_flow_style=False)


df_state = config.to_df_state()

df_state.index.name = 'grid'
df_state = df_state.rename(index={0: 1})

2024-12-03 16:04:04,359 - SuPy - INFO - All cache cleared.


In [ ]:
# class MyDumper(yaml.SafeDumper):
#     def __init__(self, *args, **kwargs):
#         super(MyDumper, self).__init__(*args, **kwargs)
#         self.depth = 0

#     def increase_indent(self, flow=False, indentless=False):
#         self.depth += 1
#         return super(MyDumper, self).increase_indent(flow, indentless)

#     def decrease_indent(self):
#         self.depth -= 1
#         return super(MyDumper, self).decrease_indent()

#     def represent_mapping(self, tag, mapping, flow_style=None):
#         self.depth += 1
#         node = super(MyDumper, self).represent_mapping(tag, mapping, flow_style)
#         self.depth -= 1
#         return node

# def represent_list_conditional(dumper, data):
#     # Check the depth of the current node
#     if dumper.depth >= 2:  # Adjust the depth as needed
#         return dumper.represent_sequence('tag:yaml.org,2002:seq', data, flow_style=True)
#     else:
#         return dumper.represent_sequence('tag:yaml.org,2002:seq', data, flow_style=False)

# yaml.add_representer(list, represent_list_conditional, Dumper=MyDumper)

# Assuming config_dict is already defined
# with open('./output.yml', 'w') as file:
#     yaml.dump(config_dict, file, Dumper=MyDumper, default_flow_style=False)

In [ ]:
# Check index and columns first
for i in range(1):
    index_equal = df_state.index.equals(df_state_init.index)
    columns_equal = df_state.columns.equals(df_state_init.columns)
    print(f"Are indices equal? {index_equal}")
    print(f"Are columns equal? {columns_equal}")

    if not index_equal or not columns_equal:
        print("Indices or columns are not equal. Cannot compare dataframes.")
        
        if not index_equal:
            # Find the differences in indices
            index_diff_df_state = df_state.index.difference(df_state_init.index)
            index_diff_df_state_init = df_state_init.index.difference(df_state.index)
            print("Indices in df_state but not in df_state_init:")
            print(index_diff_df_state)
            print("Indices in df_state_init but not in df_state:")
            print(index_diff_df_state_init)
        
        if not columns_equal:
            # Sort the columns before comparing
            sorted_columns_df_state = df_state.columns.sort_values()
            sorted_columns_df_state_init = df_state_init.columns.sort_values()
            
            # Check if sorted columns are equal
            sorted_columns_equal = sorted_columns_df_state.equals(sorted_columns_df_state_init)
            print(f"Are sorted columns equal? {sorted_columns_equal}")
            
            if not sorted_columns_equal:
                # Find the differences in columns
                columns_diff_df_state = sorted_columns_df_state.difference(sorted_columns_df_state_init)
                columns_diff_df_state_init = sorted_columns_df_state_init.difference(sorted_columns_df_state)
                print("Columns in df_state but not in df_state_init:")
                print(columns_diff_df_state)
                print("Columns in df_state_init but not in df_state:")
                print(columns_diff_df_state_init)
            else:
                # Reorder the columns in df_state to match the order in df_state_init
                print("Reordering columns in df_state to match the order in df_state_init")
                df_state = df_state.reindex(columns=df_state_init.columns)
                # df_state_init = df_state_init.reindex(columns=sorted_columns_df_state_init)


# Check if the dataframes are equal
are_equal = df_state.equals(df_state_init)
print(f"Are df_state and df_state_init equal? {are_equal}")

# If they are not equal, find the differences
if not are_equal:
    # Check for NaN values
    nan_equal = df_state.isna().equals(df_state_init.isna())
    print(f"Are NaN values equal? {nan_equal}")

    if not nan_equal:
        nan_diff = df_state.isna().compare(df_state_init.isna())
        print("Differences in NaN values:")
        print(nan_diff)

    # Check dimensions of all values within the dataframes
    dimensions_equal = df_state.applymap(lambda x: x.shape if hasattr(x, 'shape') else None).equals(
        df_state_init.applymap(lambda x: x.shape if hasattr(x, 'shape') else None))
    print(f"Are dimensions of all values equal? {dimensions_equal}")

    if not dimensions_equal:
        dimensions_diff = df_state.applymap(lambda x: x.shape if hasattr(x, 'shape') else None).compare(
            df_state_init.applymap(lambda x: x.shape if hasattr(x, 'shape') else None))
        print("Differences in dimensions of values:")
        print(dimensions_diff)

    for i in range(2):
        # Check data types
        dtypes_equal = df_state.dtypes.equals(df_state_init.dtypes)
        print(f"Are data types equal? {dtypes_equal}")

        if not dtypes_equal:
            dtypes_diff = pd.concat([df_state.dtypes, df_state_init.dtypes], axis=1, keys=["df_state", "df_state_init"])
            dtypes_diff = dtypes_diff[dtypes_diff["df_state"] != dtypes_diff["df_state_init"]]
            print("Differences in data types:")
            print(dtypes_diff)
            unique_dtypes_df_state = df_state.dtypes.unique()
            unique_dtypes_df_state_init = df_state_init.dtypes.unique()
            print("Unique data types in df_state:")
            print(unique_dtypes_df_state)
            print("Unique data types in df_state_init:")
            print(unique_dtypes_df_state_init)

            # BEGIN: make all types in df_state the same as df_state_init
            print("Making all types in df_state the same as df_state_init")
            for col in df_state.columns:
                if col in df_state_init.columns:
                    df_state[col] = df_state[col].astype(df_state_init[col].dtype)
            # END: make all types in df_state the same as df_state_init

    # Find the differences in values
    diff = df_state.compare(df_state_init)
    print("Differences between df_state and df_state_init:")
    print(diff)

    # Check for exact equality
    exact_equal = (df_state.values == df_state_init.values).all()
    print(f"Are all values exactly equal? {exact_equal}")

    # if not exact_equal:
    #     # # Find the exact differences
    #     # exact_diff = (df_state != df_state_init).stack()
    #     # print("Exact differences in values:")
    #     # print(exact_diff[exact_diff])
    #     # Set the values to be equal
    #     df_state.update(df_state_init)
    #     print("Values in df_state have been updated to match df_state_init")

Are indices equal? True
Are columns equal? False
Indices or columns are not equal. Cannot compare dataframes.
Are sorted columns equal? True
Reordering columns in df_state to match the order in df_state_init
Are df_state and df_state_init equal? False
Are NaN values equal? True
Are dimensions of all values equal? True
Are data types equal? False
Differences in data types:
                           df_state df_state_init
var                ind_dim                       
ohm_threshsw       (7,)       int64       float64
ohm_threshwd       (7,)       int64       float64
diagnose           0        float64         int64
localclimatemethod 0        float64         int64
stebbsmethod       0        float64         int64
...                             ...           ...
tsfc_surf          (2,)     float64         int64
                   (3,)     float64         int64
                   (4,)     float64         int64
                   (5,)     float64         int64
                   (6,)  

In [ ]:
df_output, df_state = supy.run_supy(df_forcing, df_state)

2024-12-03 14:23:43,146 - SuPy - INFO - ====================
2024-12-03 14:23:43,146 - SuPy - INFO - Simulation period:
2024-12-03 14:23:43,147 - SuPy - INFO -   Start: 2012-01-01 00:10:00
2024-12-03 14:23:43,147 - SuPy - INFO -   End: 2012-12-31 23:55:00
2024-12-03 14:23:43,147 - SuPy - INFO - 
2024-12-03 14:23:43,147 - SuPy - INFO - No. of grids: 1
2024-12-03 14:23:43,148 - SuPy - INFO - SuPy is running in serial mode
 Initialising STEBBS
2024-12-03 14:24:26,423 - SuPy - INFO - Execution time: 43.3 s
2024-12-03 14:24:26,424 - SuPy - INFO - ====================



In [ ]:
df_output['STEBBS']

var                             ws       Tair      Tsurf  Kroof       Lroof  \
grid datetime                                                                 
1    2012-01-01 00:10:00  0.994771  13.209780  11.772426    0.0  346.239704   
     2012-01-01 00:15:00  0.994771  13.209780  11.772426    0.0  346.239704   
     2012-01-01 00:20:00  0.994771  13.209780  11.772426    0.0  346.239704   
     2012-01-01 00:25:00  0.994771  13.209780  11.772426    0.0  346.239704   
     2012-01-01 00:30:00  0.994771  13.209780  11.772426    0.0  346.239704   
...                            ...        ...        ...    ...         ...   
     2012-12-31 23:35:00  0.712110  15.406889  10.140000    0.0  332.577203   
     2012-12-31 23:40:00  0.712110  15.687828  10.140000    0.0  332.577203   
     2012-12-31 23:45:00  0.712110  16.226356  10.140000    0.0  332.577203   
     2012-12-31 23:50:00  0.712110  16.508302  10.140000    0.0  332.577203   
     2012-12-31 23:55:00  0.712110  17.024696  10.140000    0.0  332.577203   

var                       Kwall       Lwall  Qheat_dom  Qcool_dom  dom_temp  \
grid datetime                                                                 
1    2012-01-01 00:10:00    0.0  337.192884        0.0        0.0       NaN   
     2012-01-01 00:15:00    0.0  337.192884        0.0        0.0       NaN   
     2012-01-01 00:20:00    0.0  337.192884        0.0        0.0       NaN   
     2012-01-01 00:25:00    0.0  337.192884        0.0        0.0       NaN   
     2012-01-01 00:30:00    0.0  337.192884        0.0        0.0       NaN   
...                         ...         ...        ...        ...       ...   
     2012-12-31 23:35:00    0.0  329.034697        0.0        0.0       NaN   
     2012-12-31 23:40:00    0.0  329.034697        0.0        0.0       NaN   
     2012-12-31 23:45:00    0.0  329.034697        0.0        0.0       NaN   
     2012-12-31 23:50:00    0.0  329.034697        0.0        0.0       NaN   
     2012-12-31 23:55:00    0.0  329.034697        0.0        0.0       NaN   

var                       ...  QEC  QH  QS  QBAE  QWaste  Textwallroof  \
grid datetime             ...                                            
1    2012-01-01 00:10:00  ...  NaN NaN NaN   NaN     NaN           NaN   
     2012-01-01 00:15:00  ...  NaN NaN NaN   NaN     NaN           NaN   
     2012-01-01 00:20:00  ...  NaN NaN NaN   NaN     NaN           NaN   
     2012-01-01 00:25:00  ...  NaN NaN NaN   NaN     NaN           NaN   
     2012-01-01 00:30:00  ...  NaN NaN NaN   NaN     NaN           NaN   
...                       ...  ...  ..  ..   ...     ...           ...   
     2012-12-31 23:35:00  ...  NaN NaN NaN   NaN     NaN           NaN   
     2012-12-31 23:40:00  ...  NaN NaN NaN   NaN     NaN           NaN   
     2012-12-31 23:45:00  ...  NaN NaN NaN   NaN     NaN           NaN   
     2012-12-31 23:50:00  ...  NaN NaN NaN   NaN     NaN           NaN   
     2012-12-31 23:55:00  ...  NaN NaN NaN   NaN     NaN           NaN   

var                       Tintwallroof  Textwindow  Tintwindow  Tair_ind  
grid datetime                                                             
1    2012-01-01 00:10:00           NaN         NaN         NaN       NaN  
     2012-01-01 00:15:00           NaN         NaN         NaN       NaN  
     2012-01-01 00:20:00           NaN         NaN         NaN       NaN  
     2012-01-01 00:25:00           NaN         NaN         NaN       NaN  
     2012-01-01 00:30:00           NaN         NaN         NaN       NaN  
...                                ...         ...         ...       ...  
     2012-12-31 23:35:00           NaN         NaN         NaN       NaN  
     2012-12-31 23:40:00           NaN         NaN         NaN       NaN  
     2012-12-31 23:45:00           NaN         NaN         NaN       NaN  
     2012-12-31 23:50:00           NaN         NaN         NaN       NaN  
     2012-12-31 23:55:00           NaN         NaN         NaN       NaN  

[1054